This is a compilation of examples of uses of RSRS

In [ ]:
import json
import numpy as np
import subprocess

pi = np.pi

##### Construction of the test

## More kernels can be added in the file 'structured_operators.py'

## Data Type Arguments (what operator, which precision):
structured_operator_type = ["BasicStructuredOperator","BemppClLaplaceSingleLayer", "BemppClHelmholtzSingleLayer", "KiFMMLaplaceOperator", "KiFMMHelmholtzOperator", "BemppRsLaplaceOperator"]
precision = ["Single", "Double"] # Single precision methods have not been enabled yet


data_type_args = {
    "structured_operator_type": structured_operator_type[5],
    "precision": precision[1]
}


## Scenario Arguments:
h = 0.1
id_tols = [1e-2, 1e-3, 1e-4, 1e-6, 1e-8]#[1e-2, 1e-3, 1e-4, 1e-6, 1e-8] ## A test will run for each ID tolerance
dim_args = [{"Kappa": 2*pi}, {"KappaAndMeshwidth": (pi, h)}, {"Meshwidth": h}] ## A test should run for each dim_arg, but only one test for now is enabled.
                           ## If dim_arg if of type Kappa, the algorithm will take kappa and set the 
                           # characteristic meshwidth to be h = 2.0 * pi / (8.0 * kappa) (8 points per wavelength)

                           ## Other options are: 
                           # - KappaAndMeshwidth: receives a tuple, (kappa, h), so kappa is independent of h.
                           # - MeshWidth: In case there is not an associated wavenumber and the user wants to set h.

geometry_type = ["SphereSurface", "CubeSurface", "CylinderSurface", "EllipsoidSurface"] ## We can generate more with Gmsh and link them


scenario_args = {
    "id_tols": id_tols,
    "dim_args": [dim_args[0]],
    "geometry_type": geometry_type[0],
}


## Output Arguments: (outputs that the test the test will return)
solve_tol = 1e-5
solve = [{"True": solve_tol}, {"False"}]
plot = [True, False] ## Plot time piechart indicating the percentage of time RSRS spends on each step.
dense_errors = [True, False] ## True for debugging purposes, since it performs RSRS 
                             ## on a dense matrix (dense from Python should be defined). It returns
                             ## statistics on the relative errors from LU and ID steps.
results_output = ["All", "Rank", "Time"] ## In all cases, it returns the errors ||A_app^{-1}A - I||_2 and ||A_app - A||_2/||A||_2, 
                                         ## but we can also either ask for the compression results (Rank) or time that it takes 
                                         ## to complete each RSRS step.

output_args = {
    "solve": solve[0],
    "plot": plot[0],
    "dense_errors": dense_errors[1],
    "results_output": results_output[0]
}

## RSRS arguments:

null_method = ["Projection", "Svd", "Qr"] ## Method used to nullifying matrix. 
                                          ## The cheapest way is to build the projector: I-\Omega^+\Omega,
                                          ## but SVD and QR are also allowed.
block_extraction_method = ["LuLstSq","Svd"] ## Method used to solve the least squares problem. 
                                            ## The cheapest is to solve normal equations through LU.
pivot_method = ["Lu", "DirectInversion"] ## Way to compute the pivot or the inverse of a squared matrix.
rank_picking = ["Min", ## After the leaf level, when merging boxes, it looks for the smallest skeleton size and assigns it as the new box's fixed rank.
                "Max", ## Same as before, but it takes the largest skeleton size.
                "Avg", ## Same as before, but it averages the skeleton sizes.
                "Mid", ## Same as before, but it takes the skeleton size that is in the middle of the set.
                "DoubleMin", ## Takes double the minimum rank of the previous level as a fixed rank.
                "Tol", ## Always takes the defined tolerance.
                ]

##Important: A fixed rank can be picked by selecting Tol and setting a tolerance bigger or equal to 1.

rsrs_args = {
    "oversampling": 8, ## Oversampling for each individual block
    "oversampling_diag_blocks": 16, ## Oversampling used when extracting diagonal blocks when RSRS finishes
    "initial_num_samples": 20,  ## Initial num samples: This is useful only when sampling is done in parallel way, which cannot be done atm
    "null_method": null_method[0],
    "near_block_extraction_method": block_extraction_method[0],
    "diag_block_extraction_method": block_extraction_method[0],
    "lu_pivot_method": pivot_method[0],
    "diag_pivot_method": pivot_method[0],
    "tol_null": 1e-10, ## Tolerance when nullifying blocks
    "tol_id": 1e-2, ## ID tolerance (Irrelevant, since it is set with the scenario arguments)
    "tol_ext_near": 1e-10, ## Tolerance used to compute pseudo inverses when extracting near field.
    "tol_diag_ext": 1e-16, ## Tolerance used to compute pseudo inverses when extracting diagonal blocks.
    "min_rank": 4, ## Minimum size of the box. If the box is smaller, it will be saved for the next level.
    "hermitian": True, ## Indicates if we should run RSRS for hermitian matrices (half the time and memory)
    "rank_picking": rank_picking[0] ## Rank picking strategy
}

def json_for_bash(obj):
    return json.dumps(obj).replace("'", "'\"'\"'")

data_type_args_json = json_for_bash(data_type_args)
scenario_args_json = json_for_bash(scenario_args)
output_args_json = json_for_bash(output_args)
rsrs_args_json = "null" if rsrs_args is None else json_for_bash(rsrs_args)

bash_script = f"""#!/bin/bash
export OPENBLAS_NUM_THREADS=1
cargo run --release '{data_type_args_json}' '{scenario_args_json}' '{rsrs_args_json}' '{output_args_json}'
"""

# The test is saved to the bash script run_test.sh
script_filename = "run_test.sh"
with open(script_filename, "w") as f:
    f.write(bash_script)

subprocess.run(["chmod", "+x", script_filename], check=True)



CompletedProcess(args=['chmod', '+x', 'run_test.sh'], returncode=0)

In [2]:
##### Running test in Rust

!./run_test.sh

   Compiling bempp-rsrs v0.0.1-dev (/Users/ignacia-fp/Research/WIP/Projects/rsrs-exps)
    Finished `release` profile [optimized + debuginfo] target(s) in 8.84s  
note: to see what the problems were, use the option `--future-incompat-report`, or run `cargo report future-incompatibilities --id 1`
     Running `target/release/bempp-rsrs '{"structured_operator_type": "BemppClLaplaceSingleLayer", "precision": "Double"}' '{"id_tols": [0.01, 0.0001, 1e-06, 1e-08], "dim_args": [{"KappaAndMeshwidth": [3.141592653589793, 0.1]}], "geometry_type": "SphereSurface"}' '{"oversampling": 8, "oversampling_diag_blocks": 16, "initial_num_samples": 420, "null_method": "Projection", "near_block_extraction_method": "LuLstSq", "diag_block_extraction_method": "LuLstSq", "lu_pivot_method": "Lu", "diag_pivot_method": "Lu", "tol_null": 1e-10, "tol_id": 0.01, "tol_ext_near": 1e-10, "tol_diag_ext": 1e-10, "min_rank": 4, "hermitian": true, "rank_picking": "Min"}' '{"solve": {"True": 1e-05}, "plot": true, "dense_err